# C6-B de David — 24 decisiones físicas reales por lote

Este notebook prueba una pregunta nueva y prospectiva: **¿la memoria de DMLPA aporta valor
cuando el controlador observa y actúa en 24 epochs físicos reales?**

En cada epoch el transductor exacto se detiene inmediatamente antes de la llegada de un lote
de 5,000 unidades. El agente observa el estado generado por todas las acciones, demandas y
releases anteriores; elige `0=P_H` o `1=P_C`; el lote entra al inventario y el DES avanza
hasta la siguiente llegada. No son tres bits ficticios con estado semanal repetido.

## Default `SERIOUS`

Selecciona **Run all**. Sin editar nada ejecuta cinco brazos, tres seeds y el mismo presupuesto
de 200,192 pasos por seed:

- RecurrentPPO con la arquitectura histórica MLP-LSTM: baseline en este mismo C6-B.
- PPO + DMLPA con historia física completa de 24 epochs.
- PPO + la misma DMLPA pero stack 1: ablación directa de memoria.
- RecurrentPPO + DMLPA stack 24.
- SAC categórico discreto + DMLPA stack 24.

Los controladores estructurados reciben la misma observación y la misma acción binaria. Se
elige una configuración usando tapes de selección separadas y se evalúa una sola vez en tapes
de desarrollo disjuntas. El notebook no abre seeds científicas ni modifica Program Q/Q-R1.

> **Autoridad física nueva:** elegir el destino del lote justo antes de su llegada a Op8 es
> una hipótesis C6-B de desarrollo. Requiere validación de Garrido antes de cualquier claim de
> dominio. Un PASS aquí autoriza preregistrar; no autoriza publicación ni despliegue.

**Tiempo orientativo:** extrapolando el smoke PPO medido en un Mac CPU, los cuatro brazos de
la familia PPO suman aproximadamente **12 horas** para las tres seeds `serious`. No se incluye
SAC en esa cifra: su costo depende mucho de GPU y de las actualizaciones off-policy, y puede
dominar la corrida. Para `serious`, usa GPU y evita asumir que terminará en una sesión corta.

In [ ]:
# 1 — CONFIGURACIÓN. El default ya está listo para Run all.
import ast, hashlib, importlib.metadata, inspect, json, math, os, platform, shutil, subprocess, sys, time
from collections import deque
from pathlib import Path
from typing import Any

RUN_PROFILE = os.environ.get("DAVID_C6B_PROFILE", "serious")
MODEL_KINDS = [
    "recurrent_ppo_mlp",
    "ppo_dmlpa_stack24",
    "ppo_dmlpa_stack1",
    "recurrent_ppo_dmlpa_stack24",
    "sac_discrete_dmlpa_stack24",
]
FRAME_STACK = 24
FEATURES_DIM = 120
DMLPA_HIDDEN = 100
DMLPA_HEADS = 12
DMLPA_LAYERS = 4

PROFILES = {
    "debug": dict(timesteps=768, optimizer_seeds=[9201], selection_tapes=2, eval_tapes=2),
    "screen": dict(timesteps=50_000, optimizer_seeds=[9201, 9202, 9203], selection_tapes=6, eval_tapes=8),
    "serious": dict(timesteps=200_192, optimizer_seeds=[9201, 9202, 9203], selection_tapes=12, eval_tapes=24),
}
CFG = PROFILES[RUN_PROFILE]
TOTAL_TIMESTEPS = CFG["timesteps"]
OPTIMIZER_SEEDS = CFG["optimizer_seeds"]
SELECTION_TAPES_PER_CELL = CFG["selection_tapes"]
EVAL_TAPES_PER_CELL = CFG["eval_tapes"]

TRAIN_TAPE_START, TRAIN_TAPE_END = 972_000_001, 972_099_999
SELECTION_SEEDS = list(range(982_000_001, 982_000_001 + SELECTION_TAPES_PER_CELL))
EVAL_SEEDS = list(range(983_000_001, 983_000_001 + EVAL_TAPES_PER_CELL))
ALLOWED_TAPE_RANGES = ((972_000_001, 972_099_999), (982_000_001, 982_099_999), (983_000_001, 983_099_999))

def assert_dev_tape(seed: int) -> None:
    if not any(lo <= int(seed) <= hi for lo, hi in ALLOWED_TAPE_RANGES):
        raise RuntimeError(f"TAPE PROHIBIDA: {seed}")

for tape in (TRAIN_TAPE_START, TRAIN_TAPE_END, *SELECTION_SEEDS, *EVAL_SEEDS):
    assert_dev_tape(tape)
print({"profile": RUN_PROFILE, "models": MODEL_KINDS, "timesteps_per_seed": TOTAL_TIMESTEPS,
       "optimizer_seeds": OPTIMIZER_SEEDS, "selection_tapes_per_cell": SELECTION_TAPES_PER_CELL,
       "eval_tapes_per_cell": EVAL_TAPES_PER_CELL, "frame_stack": FRAME_STACK})

In [ ]:
# 2 — Repositorio y dependencias
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "qr1-c1-natural-continuation"
CORE_COMMIT = "49f7802baedb47e0b1d23e23fa317504be059b71"
PACKAGES = [
    "simpy>=4.1", "numpy>=1.26", "gymnasium>=1.3", "stable-baselines3>=2.9",
    "sb3-contrib>=2.9", "torch>=2.1", "einops>=0.8", "pandas>=2.2",
]

def run(command):
    print("$", " ".join(map(str, command)))
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout or "")[-1600:]); print((result.stderr or "")[-1600:])
    if result.returncode:
        raise RuntimeError(command)

if IN_COLAB:
    REPO = Path("/content/scresia_c6b")
elif IN_KAGGLE:
    REPO = Path("/kaggle/working/scresia_c6b")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if IN_COLAB or IN_KAGGLE:
    run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
    if REPO.exists():
        shutil.rmtree(REPO)
    run(["git", "clone", "--depth", "20", "--branch", GIT_BRANCH, GIT_URL, str(REPO)])
    run(["git", "-C", str(REPO), "checkout", "--detach", CORE_COMMIT])
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 3 — Contrato físico C6-B embebido · ⛔ NO CAMBIAR

Esta celda contiene el entorno auditado. Para cada acción, el tiempo físico avanza hasta una
llegada de lote distinta. Al terminal, un replay independiente verifica OAT, backlog, inventario
y métricas contra el transductor vectorizado.

In [ ]:
"""Prospective C6-B environment with 24 real, causal batch decision epochs.

The frozen Program O environment chooses one of four weekly count allocations.
This development-only environment instead pauses the exact transducer immediately
before each of the 24 action-independent batch arrivals.  The controller observes
the physical state generated by every earlier action and exogenous event, chooses
``P_H`` or ``P_C`` for the arriving 5,000-unit batch, and only then advances to the
next arrival.  Terminal metrics are independently recomputed by the frozen vector
transducer and must match the incremental state.

This is a new action-authority contract, not a reinterpretation of Program Q.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Mapping, Sequence

import gymnasium as gym
from gymnasium import spaces
import numpy as np

from supply_chain.program_o_full_des import PRODUCTS
from supply_chain.program_o_full_des_transducer import (
    FullDESSkeleton,
    extract_full_des_skeleton,
    simulate_full_des_frontier,
)
from supply_chain.program_o_hobs import (
    posterior_after_week,
    predicted_request_share_c,
    transition_belief,
)
from supply_chain.program_o_ret_env import CONFIRMED_RET_CELLS, ProgramORetCell


BATCH_QUANTITY = 5_000.0
OBSERVATION_DIM = 21
PATTERN_SCHEDULER = {
    str(pattern): tuple(
        "P_C" if (pattern >> position) & 1 else "P_H" for position in range(3)
    )
    for pattern in range(8)
}


def bits_to_weekly_patterns(bits: Sequence[int]) -> tuple[int, ...]:
    values = tuple(int(value) for value in bits)
    if len(values) != 24 or any(value not in (0, 1) for value in values):
        raise ValueError("bits must contain exactly 24 binary actions")
    return tuple(
        values[3 * week]
        | (values[3 * week + 1] << 1)
        | (values[3 * week + 2] << 2)
        for week in range(8)
    )


@dataclass(frozen=True)
class PerBatchStructuredConfig:
    policy_id: str
    hysteresis: float = 0.0

    @property
    def config_id(self) -> str:
        return f"{self.policy_id}__{self.hysteresis:g}"


def structured_configurations() -> tuple[PerBatchStructuredConfig, ...]:
    """Small, finite family with the same binary action and observation."""
    return (
        PerBatchStructuredConfig("belief_pressure", 0.0),
        PerBatchStructuredConfig("belief_pressure", 2_500.0),
        PerBatchStructuredConfig("belief_pressure", 5_000.0),
        PerBatchStructuredConfig("backlog_first", 0.0),
        PerBatchStructuredConfig("inventory_balance", 0.0),
    )


def structured_action(
    raw: Mapping[str, Any], config: PerBatchStructuredConfig
) -> int:
    """Choose P_H=0 or P_C=1 from only the deployable observation payload."""
    on_hand = np.asarray(raw["on_hand"], dtype=float)
    backlog = np.asarray(raw["backlog_quantity"], dtype=float)
    in_flight = np.asarray(raw["in_flight_quantity"], dtype=float)
    position = on_hand + in_flight - backlog
    previous = raw["previous_action"]
    policy = config.policy_id
    if policy == "backlog_first":
        pressure = backlog[0] - backlog[1]
    elif policy == "inventory_balance":
        pressure = position[1] - position[0]
    elif policy == "belief_pressure":
        share_c = float(raw["predicted_share_c"])
        horizon_demand = 15_000.0 * np.asarray((share_c, 1.0 - share_c))
        pressure = (horizon_demand[0] - position[0]) - (
            horizon_demand[1] - position[1]
        )
    else:
        raise ValueError(f"unknown structured policy: {policy}")
    if previous is not None and abs(float(pressure)) <= float(config.hysteresis):
        return int(previous)
    return int(float(pressure) > 0.0)


class ProgramOPerBatchEnv(gym.Env[np.ndarray, int]):
    """Exact 24-epoch C6-B environment over one Program O skeleton."""

    metadata = {"render_modes": []}

    def __init__(
        self,
        *,
        scheduler: Mapping[str, Sequence[str]],
        tape_seed_start: int,
        tape_seed_end: int,
        cells: Sequence[ProgramORetCell] = CONFIRMED_RET_CELLS,
        belief_model_persistence: float = 0.75,
        belief_model_dominant_share: float = 0.90,
    ) -> None:
        super().__init__()
        if int(tape_seed_end) < int(tape_seed_start):
            raise ValueError("tape_seed_end must be >= tape_seed_start")
        if not cells:
            raise ValueError("at least one cell is required")
        self.scheduler = {str(key): tuple(value) for key, value in scheduler.items()}
        self.tape_seed_start = int(tape_seed_start)
        self.tape_seed_end = int(tape_seed_end)
        self.cells = tuple(cells)
        self.belief_model_persistence = float(belief_model_persistence)
        self.belief_model_dominant_share = float(belief_model_dominant_share)
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(OBSERVATION_DIM,), dtype=np.float32
        )
        self._episode_index = 0
        self._skeleton: FullDESSkeleton | None = None
        self._cell: ProgramORetCell | None = None
        self._events: list[tuple[float, int, str, Any]] = []
        self._event_index = 0
        self._current_batch: tuple[float, int, int] | None = None
        self._actions: list[int] = []
        self._now = 0.0

    def _tape_seed(self) -> int:
        count = self.tape_seed_end - self.tape_seed_start + 1
        if self._episode_index >= count:
            raise RuntimeError("C6-B training tape namespace exhausted")
        return self.tape_seed_start + self._episode_index

    def _build_skeleton(self, seed: int, cell: ProgramORetCell) -> FullDESSkeleton:
        skeleton, _ = extract_full_des_skeleton(
            seed=int(seed),
            scheduler=self.scheduler,
            regime_persistence=float(cell.regime_persistence),
            dominant_share=float(cell.dominant_share),
            downstream_freight_physics_mode="fixed_clock_physical_v1",
        )
        return skeleton

    def _initialize_state(self) -> None:
        if self._skeleton is None:
            raise RuntimeError("skeleton missing")
        skeleton = self._skeleton
        n_orders = len(skeleton.order_times)
        self._inventory = np.asarray(skeleton.opening_inventory, dtype=float).copy()
        self._pending = np.zeros(n_orders, dtype=bool)
        self._created = np.zeros(n_orders, dtype=bool)
        self._released = np.zeros(n_orders, dtype=bool)
        self._oat = np.full(n_orders, np.inf, dtype=float)
        self._bt = np.zeros(n_orders, dtype=np.uint8)
        self._requested_product = np.asarray(
            [PRODUCTS.index(value) for value in skeleton.order_products], dtype=np.int8
        )
        self._quantities = np.asarray(skeleton.order_quantities, dtype=float)
        self._opt = np.asarray(skeleton.order_times, dtype=float)
        contingent = (
            np.zeros(n_orders, dtype=bool)
            if skeleton.order_contingent is None
            else np.asarray(skeleton.order_contingent, dtype=bool)
        )
        self._priority_order = np.lexsort(
            (np.arange(n_orders), self._opt, self._quantities, 1 - contingent.astype(np.uint8))
        )
        completion = (
            tuple(float(value) + 48.0 for value in skeleton.release_slots)
            if skeleton.release_completion_slots is None
            else tuple(map(float, skeleton.release_completion_slots))
        )
        available = (
            tuple(True for _ in skeleton.release_slots)
            if skeleton.release_available is None
            else tuple(map(bool, skeleton.release_available))
        )
        self._release_completion = completion
        self._release_available = available
        events: list[tuple[float, int, str, Any]] = []
        for index, time in enumerate(skeleton.release_slots):
            events.append((float(time), 0, "release", int(index)))
        for time, week, position in skeleton.batch_arrivals:
            events.append((float(time), 1, "batch", (int(week), int(position))))
        for index, time in enumerate(skeleton.order_times):
            events.append((float(time), 2, "demand", int(index)))
        self._events = sorted(events)
        self._event_index = 0
        self._current_batch = None
        self._actions = []
        self._now = float(skeleton.decision_start)

    def _process_release(self, release_index: int) -> None:
        if not self._release_available[release_index]:
            return
        chosen = -1
        for order_index in self._priority_order:
            if not self._created[order_index] or not self._pending[order_index]:
                continue
            product = self._requested_product[order_index]
            if self._inventory[product] + 1e-9 >= self._quantities[order_index]:
                chosen = int(order_index)
                break
        if chosen < 0:
            return
        product = int(self._requested_product[chosen])
        self._inventory[product] -= float(self._quantities[chosen])
        self._pending[chosen] = False
        self._released[chosen] = True
        self._oat[chosen] = float(self._release_completion[release_index])

    def _process_demand(self, order_index: int) -> None:
        prior = np.arange(order_index)
        if len(prior):
            late = (self._opt[prior] + 48.0 <= self._now + 1e-12) & (
                self._oat[prior] > self._now + 1e-12
            )
            self._bt[order_index] = np.uint8(min(60, int(late.sum())))
        self._pending[order_index] = True
        self._created[order_index] = True

    def _advance_to_next_batch(self) -> bool:
        self._current_batch = None
        while self._event_index < len(self._events):
            now, _priority, kind, payload = self._events[self._event_index]
            self._event_index += 1
            self._now = float(now)
            if kind == "batch":
                week, position = payload
                self._current_batch = (float(now), int(week), int(position))
                return True
            if kind == "release":
                self._process_release(int(payload))
            elif kind == "demand":
                self._process_demand(int(payload))
        return False

    def _finish_events(self) -> None:
        while self._event_index < len(self._events):
            now, _priority, kind, payload = self._events[self._event_index]
            self._event_index += 1
            self._now = float(now)
            if kind == "batch":
                raise AssertionError("unprocessed batch remained after the 24th action")
            if kind == "release":
                self._process_release(int(payload))
            elif kind == "demand":
                self._process_demand(int(payload))

    def _belief(self) -> tuple[float, float]:
        if self._skeleton is None:
            raise RuntimeError("skeleton missing")
        start = float(self._skeleton.decision_start)
        current_week = min(7, max(0, int((self._now - start) // 168.0)))
        belief = 0.5
        for week in range(current_week + 1):
            lo = start + 168.0 * week
            hi = lo + 168.0
            labels = [
                self._skeleton.order_products[index]
                for index, time in enumerate(self._skeleton.order_times)
                if self._created[index] and lo <= float(time) < min(hi, self._now - 1e-12)
            ]
            if labels:
                belief = posterior_after_week(
                    belief,
                    labels,
                    dominant_share=self.belief_model_dominant_share,
                )
            if week < current_week:
                belief = transition_belief(
                    belief,
                    regime_persistence=self.belief_model_persistence,
                )
        return belief, predicted_request_share_c(
            belief, dominant_share=self.belief_model_dominant_share
        )

    def raw_observation(self) -> dict[str, Any]:
        if self._skeleton is None or self._current_batch is None:
            raise RuntimeError("reset() must be called before requesting an observation")
        _time, week, position = self._current_batch
        backlog = np.zeros(2, dtype=float)
        backlog_orders = np.zeros(2, dtype=int)
        max_age = np.zeros(2, dtype=float)
        in_flight = np.zeros(2, dtype=float)
        for index in range(len(self._opt)):
            product = int(self._requested_product[index])
            if self._pending[index]:
                backlog[product] += self._quantities[index]
                backlog_orders[product] += 1
                max_age[product] = max(max_age[product], self._now - self._opt[index])
            elif self._released[index] and self._oat[index] >= self._now - 1e-12:
                in_flight[product] += self._quantities[index]
        belief, predicted = self._belief()
        return {
            "batch_index": len(self._actions),
            "week": int(week),
            "position": int(position),
            "decision_time": float(self._now),
            "on_hand": self._inventory.copy(),
            "backlog_quantity": backlog,
            "backlog_orders": backlog_orders,
            "max_backlog_age": max_age,
            "in_flight_quantity": in_flight,
            "belief_c": float(belief),
            "predicted_share_c": float(predicted),
            "previous_action": None if not self._actions else int(self._actions[-1]),
            "remaining_decisions": 24 - len(self._actions),
        }

    def _normalized_observation(self) -> np.ndarray:
        raw = self.raw_observation()
        values: list[float] = []
        for key in ("on_hand", "backlog_quantity"):
            values.extend(float(value) / 120_000.0 for value in raw[key])
        values.extend(float(value) / 48.0 for value in raw["backlog_orders"])
        values.extend(float(value) / 1_344.0 for value in raw["max_backlog_age"])
        values.extend(float(value) / 120_000.0 for value in raw["in_flight_quantity"])
        values.extend((raw["belief_c"], raw["predicted_share_c"]))
        values.extend(
            (
                raw["batch_index"] / 23.0,
                raw["week"] / 7.0,
                raw["position"] / 2.0,
                raw["remaining_decisions"] / 24.0,
            )
        )
        previous = np.zeros(3, dtype=float)
        previous[0 if raw["previous_action"] is None else 1 + raw["previous_action"]] = 1.0
        values.extend(previous.tolist())
        # Two strictly historical demand-count channels for the current week.
        start = float(self._skeleton.decision_start) + 168.0 * raw["week"]
        counts = [0, 0]
        for index, time in enumerate(self._skeleton.order_times):
            if self._created[index] and start <= float(time) < self._now - 1e-12:
                counts[int(self._requested_product[index])] += 1
        values.extend(value / 6.0 for value in counts)
        vector = np.asarray(values, dtype=np.float32)
        if vector.shape != (OBSERVATION_DIM,):
            raise AssertionError(f"C6-B observation shape drift: {vector.shape}")
        return np.clip(vector, 0.0, 1.0)

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict[str, Any] | None = None,
    ) -> tuple[np.ndarray, dict[str, Any]]:
        super().reset(seed=seed)
        options = options or {}
        cell_index = int(options.get("cell_index", self._episode_index % len(self.cells)))
        if cell_index not in range(len(self.cells)):
            raise ValueError("cell_index outside C6-B cell set")
        self._cell = self.cells[cell_index]
        tape_seed = int(options.get("tape_seed", self._tape_seed()))
        supplied = options.get("skeleton")
        self._skeleton = (
            supplied
            if isinstance(supplied, FullDESSkeleton)
            else self._build_skeleton(tape_seed, self._cell)
        )
        self._episode_index += 1
        self._initialize_state()
        if not self._advance_to_next_batch():
            raise AssertionError("skeleton contains no batch arrivals")
        return self._normalized_observation(), {
            "tape_seed": tape_seed,
            "cell_id": self._cell.cell_id,
            "raw_observation": self.raw_observation(),
        }

    def step(
        self, action: int
    ) -> tuple[np.ndarray, float, bool, bool, dict[str, Any]]:
        if self._skeleton is None or self._current_batch is None:
            raise RuntimeError("reset() must be called before step()")
        value = int(action)
        if value not in (0, 1):
            raise ValueError("C6-B action must be 0=P_H or 1=P_C")
        product_index = 0 if value == 1 else 1
        self._inventory[product_index] += BATCH_QUANTITY
        self._actions.append(value)
        if len(self._actions) < 24:
            if not self._advance_to_next_batch():
                raise AssertionError("fewer than 24 batch epochs")
            return self._normalized_observation(), 0.0, False, False, {
                "raw_observation": self.raw_observation()
            }

        self._current_batch = None
        self._finish_events()
        patterns = bits_to_weekly_patterns(self._actions)
        direct_trace: dict[str, Any] = {}
        metrics = simulate_full_des_frontier(
            skeleton=self._skeleton,
            scheduler=PATTERN_SCHEDULER,
            calendars=np.asarray([patterns], dtype=np.uint8),
            trace_out=direct_trace,
        )
        terminal = {key: float(value[0]) for key, value in metrics.items()}
        completed = self._oat <= float(self._skeleton.score_time) + 1e-12
        direct_completed = len(self._oat) - int(round(terminal["unresolved_orders"]))
        if int(completed.sum()) != direct_completed:
            raise AssertionError("incremental/vector completed-order parity failure")
        direct_oat = np.asarray(
            [np.inf if row["OATj"] is None else float(row["OATj"]) for row in direct_trace["orders"]]
        )
        direct_bt = np.asarray(
            [int(row["ret_bt_at_request"]) for row in direct_trace["orders"]], dtype=np.uint8
        )
        if not np.array_equal(self._oat, direct_oat):
            raise AssertionError("incremental/vector OAT parity failure")
        if not np.array_equal(self._bt, direct_bt):
            raise AssertionError("incremental/vector backlog-counter parity failure")
        if not np.allclose(
            self._inventory,
            [terminal["ending_inventory_P_C"], terminal["ending_inventory_P_H"]],
            atol=1e-12,
            rtol=0.0,
        ):
            raise AssertionError("incremental/vector ending-inventory parity failure")
        observation = np.zeros(self.observation_space.shape, dtype=np.float32)
        return observation, float(terminal["ret_visible"]), True, False, {
            "actions": list(self._actions),
            "weekly_patterns": list(patterns),
            "metrics": terminal,
        }

In [ ]:
# 4 — Entorno + historia con máscara explícita de padding
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from einops import rearrange
from scripts.evaluate_program_o_ret_learner import scheduler

SCHED = scheduler()

class MaskedHistoryStack(gym.Wrapper):
    '''Apila observaciones y añade un bit de validez por token.'''
    def __init__(self, env: gym.Env, history_length: int):
        super().__init__(env)
        self.history_length = int(history_length)
        self.base_dim = int(env.observation_space.shape[0])
        self.history = deque(maxlen=self.history_length)
        token_low = np.concatenate([env.observation_space.low, [0.0]]).astype(np.float32)
        token_high = np.concatenate([env.observation_space.high, [1.0]]).astype(np.float32)
        self.observation_space = gym.spaces.Box(
            np.tile(token_low, self.history_length),
            np.tile(token_high, self.history_length),
            dtype=np.float32,
        )

    def _token(self, observation, valid=1.0):
        return np.concatenate([np.asarray(observation, np.float32), [valid]]).astype(np.float32)

    def _stack(self):
        return np.concatenate(tuple(self.history)).astype(np.float32, copy=False)

    def reset(self, **kwargs):
        observation, info = self.env.reset(**kwargs)
        self.history.clear()
        for _ in range(self.history_length - 1):
            self.history.append(self._token(np.zeros_like(observation), valid=0.0))
        self.history.append(self._token(observation, valid=1.0))
        return self._stack(), info

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        self.history.append(self._token(observation, valid=1.0))
        return self._stack(), reward, terminated, truncated, info

def history_for(model_kind: str) -> int:
    if model_kind in {"recurrent_ppo_mlp", "ppo_dmlpa_stack1"}:
        return 1
    return FRAME_STACK

def make_env(model_kind: str, start=TRAIN_TAPE_START, end=TRAIN_TAPE_END):
    raw = ProgramOPerBatchEnv(
        scheduler=SCHED, tape_seed_start=start, tape_seed_end=end
    )
    return MaskedHistoryStack(raw, history_for(model_kind))

probe = make_env("ppo_dmlpa_stack24")
obs, info = probe.reset(options={"tape_seed": EVAL_SEEDS[0], "cell_index": 2})
times = [info["raw_observation"]["decision_time"]]
for _ in range(23):
    obs, _, done, _, info = probe.step(0)
    if not done:
        times.append(info["raw_observation"]["decision_time"])
assert len(times) == 24 and all(b > a for a, b in zip(times, times[1:]))
print("PRECHECK OK: 24 epochs físicos distintos; obs raw=21; token DMLPA=22; stack=", FRAME_STACK)

## 5 — ARQUITECTURA DE DAVID · ✅ PUEDES EDITAR AQUÍ

Puedes revisar y modificar embedding, atención, dimensión y capas. La máscara impide que el
Transformer atienda a frames de padding. No cambies el bit final de validez ni el contrato de
observación del entorno.

In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class DavidDMLPAPositional(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor=24, features_dim=120,
                 hidden_dim=100, nhead=12, num_layers=4):
        super().__init__(observation_space, features_dim)
        self.factor = int(factor)
        flat_dim = int(observation_space.shape[0])
        if flat_dim % self.factor:
            raise ValueError("observation dimension must be divisible by factor")
        self.token_dim = flat_dim // self.factor
        if self.token_dim != OBSERVATION_DIM + 1:
            raise ValueError("each token must contain raw observation + validity bit")
        if features_dim % nhead:
            raise ValueError("features_dim must be divisible by nhead")
        self.embedding = torch.nn.Sequential(
            torch.nn.Linear(OBSERVATION_DIM, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        layer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim, nhead=nhead, batch_first=True
        )
        self.transformer = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.register_buffer("positional", self._sinusoidal(self.factor, features_dim))

    @staticmethod
    def _sinusoidal(length, width):
        pe = torch.zeros(length, width)
        position = torch.arange(length, dtype=torch.float32).unsqueeze(1)
        scale = torch.exp(torch.arange(0, width, 2) * (-math.log(10000.0) / width))
        pe[:, 0::2] = torch.sin(position * scale)
        pe[:, 1::2] = torch.cos(position * scale)
        return pe.unsqueeze(0)

    def forward(self, observations):
        tokens = rearrange(observations, "b (t d) -> b t d", t=self.factor)
        valid = tokens[:, :, -1] > 0.5
        embedded = self.pre_norm(self.embedding(tokens[:, :, :-1]) + self.positional)
        encoded = self.transformer(embedded, src_key_padding_mask=~valid)
        return encoded[:, -1, :]

print("DavidDMLPAPositional cargada. Esta es la única clase de arquitectura que David debe editar.")

In [ ]:
# 6 — Agentes: mismas acciones, información, tapes y presupuesto
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from scripts.discrete_sac_dmlpa import DiscreteSACAgent, DiscreteSACConfig

def dmlpa_policy_kwargs(model_kind):
    return {
        "features_extractor_class": DavidDMLPAPositional,
        "features_extractor_kwargs": {
            "factor": history_for(model_kind), "features_dim": FEATURES_DIM,
            "hidden_dim": DMLPA_HIDDEN, "nhead": DMLPA_HEADS,
            "num_layers": DMLPA_LAYERS,
        },
        "net_arch": dict(pi=[128, 64], vf=[128, 64]),
    }

def build_agent(model_kind, optimizer_seed):
    env = make_env(model_kind)
    common = dict(env=env, seed=int(optimizer_seed), verbose=0, learning_rate=3e-4, gamma=0.99)
    if model_kind in {"ppo_dmlpa_stack24", "ppo_dmlpa_stack1"}:
        return PPO("MlpPolicy", n_steps=768, batch_size=96, gae_lambda=0.95,
                   ent_coef=0.01, policy_kwargs=dmlpa_policy_kwargs(model_kind), **common)
    if model_kind == "recurrent_ppo_mlp":
        return RecurrentPPO(
            "MlpLstmPolicy", n_steps=768, batch_size=96, gae_lambda=0.95,
            ent_coef=0.01,
            policy_kwargs=dict(lstm_hidden_size=64, net_arch=dict(pi=[64, 64], vf=[64, 64])),
            **common,
        )
    if model_kind == "recurrent_ppo_dmlpa_stack24":
        kwargs = dmlpa_policy_kwargs(model_kind); kwargs["lstm_hidden_size"] = 64
        return RecurrentPPO("MlpLstmPolicy", n_steps=768, batch_size=96,
                            gae_lambda=0.95, ent_coef=0.01, policy_kwargs=kwargs, **common)
    if model_kind == "sac_discrete_dmlpa_stack24":
        kwargs = dmlpa_policy_kwargs(model_kind)["features_extractor_kwargs"]
        learning_starts = 744 if RUN_PROFILE == "debug" else 2_000
        return DiscreteSACAgent(
            env=env, seed=int(optimizer_seed), features_dim=FEATURES_DIM,
            extractor_factory=lambda: DavidDMLPAPositional(env.observation_space, **kwargs),
            config=DiscreteSACConfig(buffer_size=100_000, batch_size=256,
                                     learning_starts=learning_starts, learning_rate=3e-4),
        )
    raise ValueError(model_kind)

architecture_reports = {}
for kind in MODEL_KINDS:
    agent = build_agent(kind, OPTIMIZER_SEEDS[0])
    policy = getattr(agent, "policy", agent)
    architecture_reports[kind] = {
        "policy": type(policy).__name__,
        "history": history_for(kind),
        "parameters": int(sum(p.numel() for p in policy.parameters())),
        "action_space": str(agent.get_env().action_space),
    }
display(pd.DataFrame(architecture_reports).T)

In [ ]:
# 7 — Entrenamiento multiseed
RUN_ROOT = REPO / "outputs" / "david_c6b" / f"{RUN_PROFILE}_{int(time.time())}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
agents = {}; training_rows = []
for kind in MODEL_KINDS:
    for optimizer_seed in OPTIMIZER_SEEDS:
        print(f"\nEntrenando {kind} seed={optimizer_seed} pasos={TOTAL_TIMESTEPS:,}")
        started = time.time(); agent = build_agent(kind, optimizer_seed)
        agent.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=False)
        elapsed = time.time() - started
        agents[(kind, optimizer_seed)] = agent
        training_rows.append({"model": kind, "seed": optimizer_seed,
                              "timesteps": TOTAL_TIMESTEPS, "elapsed_seconds": elapsed})
        suffix = ".pt" if kind.startswith("sac_discrete") else ".zip"
        agent.save(RUN_ROOT / f"{kind}_seed{optimizer_seed}{suffix}")
        print(f"listo: {elapsed/60:.1f} min")
display(pd.DataFrame(training_rows))

In [ ]:
# 8 — Selección separada del mejor controlador estructurado
def rollout_structured(config, skeleton, cell_index, tape_seed):
    env = ProgramOPerBatchEnv(scheduler=SCHED, tape_seed_start=tape_seed, tape_seed_end=tape_seed)
    _, _ = env.reset(options={"skeleton": skeleton, "tape_seed": tape_seed, "cell_index": cell_index})
    done = False
    while not done:
        action = structured_action(env.raw_observation(), config)
        _, _, done, _, info = env.step(action)
    return info["metrics"]

structured_configs = structured_configurations()
selection_rows = []
selection_skeletons = {}
for cell_index, cell in enumerate(CONFIRMED_RET_CELLS):
    for tape_seed in SELECTION_SEEDS:
        skeleton, _ = extract_full_des_skeleton(
            seed=tape_seed, scheduler=SCHED,
            regime_persistence=cell.regime_persistence, dominant_share=cell.dominant_share,
            downstream_freight_physics_mode="fixed_clock_physical_v1")
        selection_skeletons[(cell_index, tape_seed)] = skeleton
        for config in structured_configs:
            metrics = rollout_structured(config, skeleton, cell_index, tape_seed)
            selection_rows.append({"cell": cell.cell_id, "seed": tape_seed,
                                   "config": config.config_id, "ret_visible": metrics["ret_visible"]})
selection_df = pd.DataFrame(selection_rows)
means = selection_df.groupby("config").ret_visible.mean().sort_values(ascending=False)
BEST_STRUCTURED_ID = str(means.index[0])
BEST_STRUCTURED = next(c for c in structured_configs if c.config_id == BEST_STRUCTURED_ID)
print("MEJOR ESTRUCTURADO SELECCIONADO SOLO EN TAPES SEPARADAS:", BEST_STRUCTURED_ID)
display(means.to_frame("mean_selection_ret"))

In [ ]:
# 9 — Evaluación emparejada: misma física, acción, observación y tapes
def rollout_agent(agent, model_kind, skeleton, cell_index, tape_seed):
    env = make_env(model_kind, tape_seed, tape_seed)
    observation, _ = env.reset(options={"skeleton": skeleton, "tape_seed": tape_seed,
                                        "cell_index": cell_index})
    state = None; episode_start = np.ones((1,), dtype=bool); done = False
    while not done:
        prediction = agent.predict(observation, state=state,
                                   episode_start=episode_start, deterministic=True)
        action, state = prediction if isinstance(prediction, tuple) else (prediction, None)
        observation, _, done, _, info = env.step(int(np.asarray(action).reshape(-1)[0]))
        episode_start[:] = done
    return info["metrics"]

evaluation_rows = []
RESOURCE_KEYS = ("gross_policy_batch_slots", "gross_production_quantity",
                 "charged_daily_dispatch_slots", "charged_downstream_vehicle_hours")
for cell_index, cell in enumerate(CONFIRMED_RET_CELLS):
    for tape_seed in EVAL_SEEDS:
        print("evaluando", cell.cell_id, tape_seed)
        skeleton, _ = extract_full_des_skeleton(
            seed=tape_seed, scheduler=SCHED,
            regime_persistence=cell.regime_persistence, dominant_share=cell.dominant_share,
            downstream_freight_physics_mode="fixed_clock_physical_v1")
        structured = rollout_structured(BEST_STRUCTURED, skeleton, cell_index, tape_seed)
        for kind in MODEL_KINDS:
            for optimizer_seed in OPTIMIZER_SEEDS:
                metrics = rollout_agent(agents[(kind, optimizer_seed)], kind, skeleton,
                                        cell_index, tape_seed)
                row = {"model": kind, "optimizer_seed": optimizer_seed,
                       "cell": cell.cell_id, "tape_seed": tape_seed,
                       "ret_visible": metrics["ret_visible"],
                       "delta_structured": metrics["ret_visible"] - structured["ret_visible"],
                       "worst_delta_structured": metrics["worst_product_fill"] - structured["worst_product_fill"]}
                for key in RESOURCE_KEYS:
                    row[f"resource_delta::{key}"] = metrics[key] - structured[key]
                evaluation_rows.append(row)
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df.head())

In [ ]:
# 10 — Veredicto claro: RecurrentPPO, estructurado y memoria
BOOTSTRAP_RESAMPLES = 5_000
def two_way_lcb(delta, seed=20260722):
    delta = np.asarray(delta, dtype=float)
    rng = np.random.default_rng(seed); sims = np.empty(BOOTSTRAP_RESAMPLES)
    for i in range(BOOTSTRAP_RESAMPLES):
        si = rng.integers(0, delta.shape[0], delta.shape[0])
        ti = rng.integers(0, delta.shape[1], delta.shape[1])
        sims[i] = delta[np.ix_(si, ti)].mean()
    return float(np.quantile(sims, 0.05))

def matrix(model, cell, column):
    subset = evaluation_df[(evaluation_df.model == model) & (evaluation_df.cell == cell)]
    return subset.pivot(index="optimizer_seed", columns="tape_seed", values=column).to_numpy()

verdict_rows = []
candidates = [kind for kind in MODEL_KINDS if kind != "recurrent_ppo_mlp"]
for kind in candidates:
    for cell in [c.cell_id for c in CONFIRMED_RET_CELLS]:
        candidate_ret = matrix(kind, cell, "ret_visible")
        recurrent_ret = matrix("recurrent_ppo_mlp", cell, "ret_visible")
        delta_rppo = candidate_ret - recurrent_ret
        delta_structured = matrix(kind, cell, "delta_structured")
        worst = matrix(kind, cell, "worst_delta_structured")
        resource_cols = [c for c in evaluation_df.columns if c.startswith("resource_delta::")]
        subset = evaluation_df[(evaluation_df.model == kind) & (evaluation_df.cell == cell)]
        resource_max = max(
            float(np.max(np.abs(subset[column].to_numpy(dtype=float))))
            for column in resource_cols
        )
        row = {"model": kind, "cell": cell,
               "delta_vs_RecurrentPPO": float(delta_rppo.mean()),
               "delta_vs_RecurrentPPO_LCB05": two_way_lcb(delta_rppo),
               "delta_vs_structured": float(delta_structured.mean()),
               "delta_vs_structured_LCB05": two_way_lcb(delta_structured, 20260723),
               "worst_product_delta_LCB05": two_way_lcb(worst, 20260724),
               "resource_max_abs": resource_max}
        row["strong_cell_pass"] = bool(
            row["delta_vs_RecurrentPPO_LCB05"] > 0.0
            and row["delta_vs_structured_LCB05"] >= 0.01
            and row["worst_product_delta_LCB05"] >= -0.02
            and resource_max == 0.0)
        verdict_rows.append(row)

verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)
final_verdict = {}
for kind in candidates:
    rows = verdict_df[verdict_df.model == kind]
    final_verdict[kind] = {
        "beat_recurrent_ppo_all_cells": bool((rows.delta_vs_RecurrentPPO_LCB05 > 0).all()),
        "beat_structured_plus_0p01_all_cells": bool((rows.delta_vs_structured_LCB05 >= 0.01).all()),
        "strong_goal_all_cells": bool(rows.strong_cell_pass.all()),
    }

# La prueba directa de memoria mantiene PPO y DMLPA idénticos; solo cambia stack 24 vs 1.
memory_rows = []
for cell in [c.cell_id for c in CONFIRMED_RET_CELLS]:
    stack24 = matrix("ppo_dmlpa_stack24", cell, "ret_visible")
    stack1 = matrix("ppo_dmlpa_stack1", cell, "ret_visible")
    delta = stack24 - stack1
    memory_rows.append({"cell": cell, "memory_delta": float(delta.mean()),
                        "memory_delta_LCB05": two_way_lcb(delta, 20260725),
                        "memory_helped": bool(two_way_lcb(delta, 20260725) > 0.0)})
memory_df = pd.DataFrame(memory_rows); display(memory_df)
final_verdict["DMLPA_MEMORY"] = {
    "helped_all_cells": bool(memory_df.memory_helped.all()),
    "interpretation": "stack24 beat the otherwise identical stack1 PPO only if all paired LCB05 values exceed zero",
}
any_strong = any(
    value.get("strong_goal_all_cells", False)
    for value in final_verdict.values() if isinstance(value, dict)
)
if RUN_PROFILE != "serious":
    RUN_OUTCOME = "SMOKE_ONLY_NO_SCIENTIFIC_CONCLUSION"
elif any_strong and final_verdict["DMLPA_MEMORY"]["helped_all_cells"]:
    RUN_OUTCOME = "C6B_DEVELOPMENT_PASS_TO_PREREGISTRATION"
else:
    RUN_OUTCOME = "C6B_DEVELOPMENT_NO_GO_UNDER_TESTED_ENVELOPE"
print("\nVEREDICTO FINAL")
print(json.dumps(final_verdict, indent=2))
print("RESULTADO GLOBAL:", RUN_OUTCOME)
print("Incluso un PASS solo autoriza preregistrar; NO es claim científico ni validación Garrido.")

In [ ]:
# 11 — Guardar bundle auditable
report = {"status": "C6B_DEVELOPMENT_ONLY_NOT_PROMOTABLE", "profile": RUN_PROFILE,
          "models": MODEL_KINDS, "timesteps_per_seed": TOTAL_TIMESTEPS,
          "optimizer_seeds": OPTIMIZER_SEEDS, "frame_stack": FRAME_STACK,
          "best_structured": BEST_STRUCTURED_ID, "architecture": architecture_reports,
          "training": training_rows, "verdict": verdict_rows,
          "memory_ablation": memory_rows, "final_verdict": final_verdict,
          "run_outcome": RUN_OUTCOME,
          "claim_boundary": "New per-batch authority; requires Garrido face validation and a fresh preregistered run."}
evaluation_df.to_csv(RUN_ROOT / "evaluation_rows.csv", index=False)
(RUN_ROOT / "development_report.json").write_text(json.dumps(report, indent=2, default=str) + "\n")
artifacts = sorted(path for path in RUN_ROOT.iterdir() if path.is_file())
checksums = [f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.name}" for path in artifacts]
(RUN_ROOT / "files.sha256").write_text("\n".join(checksums) + "\n")
bundle = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
print("reporte:", RUN_ROOT / "development_report.json")
print("bundle:", bundle)

## Instrucciones finales para David

### ✅ PUEDES CAMBIAR

- La clase `DavidDMLPAPositional`.
- `FEATURES_DIM`, hidden size, heads y capas, conservando dimensiones compatibles.
- `MODEL_KINDS` para depuración y `RUN_PROFILE="debug"` para un smoke.
- Después del smoke, vuelve a `serious` y usa **Run all**.

### ⛔ NO CAMBIES

- La física C6-B, el momento de decisión, reward, observación, máscaras o tapes.
- Los namespaces `972*`, `982*`, `983*` ni ninguna seed científica/reservada.
- El controlador estructurado después de ver evaluación: se selecciona únicamente en tapes separadas.
- Las acciones, información, presupuesto o métricas de un brazo sin cambiarlos para todos.
- La lógica del veredicto ni el margen `+0.01` después de observar resultados.

### Qué debe ganar

1. RecurrentPPO entrenado desde cero en **este mismo entorno C6-B**.
2. El mejor controlador estructurado con la misma acción binaria e información.
3. La DMLPA stack 24 debe superar a la misma DMLPA stack 1 para atribuir valor a memoria.
4. Debe conservar peor-producto y recursos exactos.

Si stack 24 no supera stack 1, DMLPA tiene capacidad de memoria pero la historia no produjo
valor incremental bajo este contrato. Si gana, sigue siendo evidencia de desarrollo: la nueva
autoridad de asignar cada lote debe validarse con Garrido antes de una corrida científica.